<a href="https://colab.research.google.com/github/NaavyaV/Accenture-1K-contract-review-challenge/blob/dev/Naavya_%2B_Yutika_Risk_Scorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -q "transformers>=4.45,<5" torch numpy pandas scikit-learn


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


Import libraries and configure the model

In [2]:

import torch
import numpy as np

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_NAME = "distilbert/distilroberta-base"
NUM_LABELS = 41

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)


Using device: cuda


Initialize DistilRoBERTa

In [3]:

def initialize_model(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS
):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="multi_label_classification"
    )

    model = model.to(DEVICE)

    return model, tokenizer


model, tokenizer = initialize_model()

print("Model initialized successfully!")
print("Number of labels:", model.config.num_labels)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilbert/distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model initialized successfully!
Number of labels: 41


Create the dataset class

In [4]:

class ContractDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length=256):
        assert len(texts) == len(labels)

        labels = np.asarray(labels, dtype=np.float32)

        assert labels.ndim == 2
        assert labels.shape[1] == NUM_LABELS

        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(values[idx])
            for key, values in self.encodings.items()
        }

        item["labels"] = self.labels[idx]

        return item


Create reusable training function

In [5]:

def train_model(
    model,
    train_loader,
    epochs=3,
    learning_rate=2e-5
):
    optimizer = AdamW(
        model.parameters(),
        lr=learning_rate
    )

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for batch in train_loader:

            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            optimizer.zero_grad()

            outputs = model(**batch)

            loss = outputs.loss

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"- Loss: {avg_loss:.4f}"
        )

    return model


Create reusable prediction function

In [6]:

def predict_clauses(
    text,
    model,
    tokenizer,
    threshold=0.5
):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)

        probabilities = torch.sigmoid(
            outputs.logits
        )

    predictions = (
        probabilities >= threshold
    ).int()

    return {
        "probabilities": probabilities.cpu().numpy()[0],
        "predictions": predictions.cpu().numpy()[0]
    }


Create example training data

In [7]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import DataLoader

# 1. Load the three files uploaded individually to Colab
train_chunks = pd.read_json("/content/cuad_train_split.jsonl", lines=True)
val_chunks = pd.read_json("/content/cuad_val_split.jsonl", lines=True)
test_chunks = pd.read_json("/content/cuad_test_split.jsonl", lines=True)

# 2. Create a consistent mapping for all 41 categories
categories = sorted({
    category
    for labels in train_chunks["labels"]
    for category in labels
})

assert len(categories) == NUM_LABELS, (
    f"Expected {NUM_LABELS} categories, found {len(categories)}"
)

label_encoder = MultiLabelBinarizer(classes=categories)
label_encoder.fit([[]])

y_train = label_encoder.transform(train_chunks["labels"])
y_val = label_encoder.transform(val_chunks["labels"])
y_test = label_encoder.transform(test_chunks["labels"])

# 3. Build datasets using the ContractDataset class
train_dataset = ContractDataset(
    train_chunks["chunk_text"].fillna("").tolist(),
    y_train,
    tokenizer,
    max_length=512
)

val_dataset = ContractDataset(
    val_chunks["chunk_text"].fillna("").tolist(),
    y_val,
    tokenizer,
    max_length=512
)

test_dataset = ContractDataset(
    test_chunks["chunk_text"].fillna("").tolist(),
    y_test,
    tokenizer,
    max_length=512
)

# 4. Create data loaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# 5. Verify everything loaded
print("Training chunks:", len(train_dataset))
print("Validation chunks:", len(val_dataset))
print("Test chunks:", len(test_dataset))
print("Number of categories:", len(categories))
print("Label vector shape:", y_train.shape)

Training chunks: 11500
Validation chunks: 1998
Test chunks: 1226
Number of categories: 41
Label vector shape: (11500, 41)


In [8]:

import numpy as np
import pandas as pd
import re

splits = {
    "train": (train_chunks, np.asarray(y_train)),
    "validation": (val_chunks, np.asarray(y_val)),
    "test": (test_chunks, np.asarray(y_test))
}

# 1. Contract leakage
ids = {
    name: set(df["contract_id"].astype(str))
    for name, (df, _) in splits.items()
}

print("CONTRACT OVERLAP")
for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    overlap = ids[a] & ids[b]
    print(f"{a} vs {b}: {len(overlap)} shared contracts")
    if overlap:
        print("Examples:", list(overlap)[:5])

# 2. Label distribution
rows = []

for j, category in enumerate(categories):
    row = {"category": category}
    for name, (_, labels) in splits.items():
        row[name + "_positives"] = int(labels[:, j].sum())
    rows.append(row)

freq = pd.DataFrame(rows).sort_values("train_positives")
display(freq)

# 3. Exact duplicate text across splits
def normalized_texts(df):
    return set(
        df["chunk_text"].fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    ) - {""}

texts = {
    name: normalized_texts(df)
    for name, (df, _) in splits.items()
}

print("\nEXACT TEXT OVERLAP")
for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    print(a, b, len(texts[a] & texts[b]))

# 4. Positive examples for manual inspection
rare = freq.head(8)["category"].tolist()

for category in rare:
    j = list(categories).index(category)
    indices = np.flatnonzero(np.asarray(y_train)[:, j] == 1)

    print("\n" + "=" * 65)
    print(category, "| training positives:", len(indices))

    for i in indices[:2]:
        row = train_chunks.iloc[int(i)]
        print("Contract:", row["contract_id"])
        print("Labels:", row["labels"])
        print("TEXT:")
        print(str(row["chunk_text"])[:1200])
        print("-" * 50)

freq.to_csv("/content/cuad_label_audit.csv", index=False)
print("\nSaved: /content/cuad_label_audit.csv")


CONTRACT OVERLAP
train vs validation: 0 shared contracts
train vs test: 0 shared contracts
validation vs test: 0 shared contracts


,category,train_positives,validation_positives,test_positives
30,Price Restrictions,20,3,1
38,Unlimited/All-You-Can-Eat-License,22,2,1
34,Source Code Escrow,25,7,0
21,Most Favored Nation,25,6,4
22,No-Solicit Of Customers,33,8,8
36,Third Party Beneficiary,33,4,4
25,Non-Disparagement,38,8,7
1,Affiliate License-Licensor,39,3,11
23,No-Solicit Of Employees,56,7,7
17,Joint Ip Ownership,63,6,7



EXACT TEXT OVERLAP
train validation 1
train test 7
validation test 0

Price Restrictions | training positives: 20
Contract: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
Labels: ['Price Restrictions']
TEXT:
                       Section   2.4(a)  shall  be  subject  to   adjustment                            annually  on the  first  day  of  each  Product  Year                            beginning in the calendar  year 2000 and on the first                            day of each succeeding Product Year for the remainder                            of the Term and all  renewals  of this  Agreement  in                            proportion   to  the  increase  or  decrease  in  the                            Consumer  Price Index (CPI) as compared to the CPI as                            it  existed  on the  first  day of the  Term  of this                            Agreement.  The Company  also  reserves  the right to                            increase  or  decrease  the price 

Train the model

In [11]:

import os, time, json, random, gc
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, get_linear_schedule_with_warmup
)
from sklearn.metrics import f1_score

# CONFIG
MODEL_NAME = "microsoft/deberta-v3-base"
ROOT = "/content/cuad_deberta_fast"
BEST_PATH = ROOT + "/best_model"

BATCH_SIZE = 4
ACCUM = 2
LR = 2e-5
MAX_STEPS = 2200
EVAL_EVERY = 600
BUDGET_SECONDS = 540  # 9 minutes; allow time to save

os.makedirs(ROOT, exist_ok=True)
assert torch.cuda.is_available()
DEVICE = torch.device("cuda")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Speed optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

tokenizer_fast = AutoTokenizer.from_pretrained(
    MODEL_NAME, use_fast=True
)

# Pre-tokenize once, instead of every batch
class FastDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer_fast(
            list(texts),
            truncation=True,
            max_length=512,
            padding=False
        )
        self.labels = np.asarray(labels, dtype=np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx]
        }

collator = DataCollatorWithPadding(
    tokenizer_fast,
    pad_to_multiple_of=8
)

def collate(batch):
    labels = torch.tensor(
        np.stack([x["labels"] for x in batch]),
        dtype=torch.float32
    )
    inputs = collator([
        {
            "input_ids": x["input_ids"],
            "attention_mask": x["attention_mask"]
        }
        for x in batch
    ])
    inputs["labels"] = labels
    return inputs

train_data = FastDataset(
    train_chunks["chunk_text"].fillna("").tolist(),
    y_train
)
val_data = FastDataset(
    val_chunks["chunk_text"].fillna("").tolist(),
    y_val
)

train_loader = DataLoader(
    train_data, batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate,
    pin_memory=True
)
val_loader = DataLoader(
    val_data, batch_size=16,
    shuffle=False, collate_fn=collate,
    pin_memory=True
)

gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(categories),
    problem_type="multi_label_classification"
).to(DEVICE)

# Stronger rare-label weighting, capped
y = np.asarray(y_train)
pos = y.sum(axis=0)
neg = len(y) - pos

weights = np.clip(
    np.sqrt(neg / np.maximum(pos, 1)),
    1, 5
)

loss_fn = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        weights, dtype=torch.float32, device=DEVICE
    )
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01
)

total_updates = MAX_STEPS // ACCUM

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=100,
    num_training_steps=total_updates
)

scaler = torch.amp.GradScaler("cuda")

@torch.no_grad()
def evaluate():
    model.eval()
    all_probs = []

    for batch in val_loader:
        inputs = {
            k: v.to(DEVICE)
            for k, v in batch.items()
            if k != "labels"
        }

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**inputs).logits

        all_probs.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )

    probs = np.concatenate(all_probs)
    labels = np.asarray(y_val).astype(int)

    # Vectorized threshold search
    thresholds = np.arange(0.10, 0.801, 0.025)
    best = {"micro_f1": -1}

    for t in thresholds:
        pred = probs >= t
        tp = np.logical_and(pred, labels == 1).sum()
        fp = np.logical_and(pred, labels == 0).sum()
        fn = np.logical_and(~pred, labels == 1).sum()

        micro = 2 * tp / max(2 * tp + fp + fn, 1)

        if micro > best["micro_f1"]:
            best = {
                "threshold": float(t),
                "micro_f1": float(micro),
                "macro_f1": float(f1_score(
                    labels, pred,
                    average="macro",
                    zero_division=0
                ))
            }

    model.train()
    return best

def save_best(metrics, step):
    model.save_pretrained(
        BEST_PATH, safe_serialization=True
    )
    tokenizer_fast.save_pretrained(BEST_PATH)

    with open(BEST_PATH + "/metadata.json", "w") as f:
        json.dump({
            "step": step,
            "metrics": metrics,
            "categories": list(categories)
        }, f, indent=2)

start = time.monotonic()
step = 0
best_micro = -1
history = []
last_eval_step = 0

model.train()
optimizer.zero_grad(set_to_none=True)

print("GPU:", torch.cuda.get_device_name(0))
print("Training started")

stop = False

while not stop:
    for batch in train_loader:
        elapsed = time.monotonic() - start
        if elapsed >= BUDGET_SECONDS:
            stop = True
            break

        inputs = {
          k: v.to(DEVICE)
          for k, v in batch.items()
          if k != "labels"
        }
        labels = batch["labels"].to(DEVICE)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**inputs).logits
            loss = loss_fn(
                logits.float(), labels
            ) / ACCUM

        scaler.scale(loss).backward()
        step += 1

        if step % ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        if step % 100 == 0:
            print(
                f"Step {step} | "
                f"Loss {loss.item()*ACCUM:.4f} | "
                f"Time {elapsed/60:.1f}m"
            )

        if step % EVAL_EVERY == 0:
            metrics = evaluate()
            last_eval_step = step

            history.append({
                "step": step, **metrics
            })
            print("VALIDATION:", metrics)

            if metrics["micro_f1"] > best_micro:
                best_micro = metrics["micro_f1"]
                save_best(metrics, step)
                print("Best checkpoint saved")

        if step >= MAX_STEPS:
            stop = True
            break

# Evaluate final model if time allows
if step != last_eval_step and (
    time.monotonic() - start < BUDGET_SECONDS - 45
    or best_micro < 0
):
    metrics = evaluate()
    history.append({"step": step, **metrics})

    if metrics["micro_f1"] > best_micro:
        best_micro = metrics["micro_f1"]
        save_best(metrics, step)

pd.DataFrame(history).to_csv(
    ROOT + "/history.csv", index=False
)

print("\nFINISHED")
print("Steps:", step)
print("Best micro-F1:", best_micro)
print("Checkpoint:", BEST_PATH)
print("Elapsed minutes:",
      round((time.monotonic()-start)/60, 2))


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


GPU: Tesla T4
Training started
Step 100 | Loss 0.5923 | Time 0.4m
Step 200 | Loss 0.2208 | Time 0.9m
Step 300 | Loss 0.2925 | Time 1.3m
Step 400 | Loss 0.1442 | Time 1.8m
Step 500 | Loss 0.3261 | Time 2.2m
Step 600 | Loss 0.0736 | Time 2.6m
VALIDATION: {'threshold': 0.175, 'micro_f1': 0.15523284927391087, 'macro_f1': 0.05751668490512752}
Best checkpoint saved
Step 700 | Loss 0.4284 | Time 5.4m
Step 800 | Loss 0.0619 | Time 5.8m
Step 900 | Loss 0.3551 | Time 6.3m
Step 1000 | Loss 0.2525 | Time 6.7m
Step 1100 | Loss 0.3144 | Time 7.2m
Step 1200 | Loss 0.2350 | Time 7.6m
VALIDATION: {'threshold': 0.44999999999999996, 'micro_f1': 0.2922289436947417, 'macro_f1': 0.07651640398274148}
Best checkpoint saved

FINISHED
Steps: 1200
Best micro-F1: 0.2922289436947417
Checkpoint: /content/cuad_deberta_fast/best_model
Elapsed minutes: 10.87


In [12]:

import os, gc, json, time, random, shutil
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score

# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
MODEL_NAME = "distilbert/distilroberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 8

# 10-minute approximate budget
TRAIN_TIME_LIMIT = 525
EVAL_EVERY = 400
MAX_STEPS = 5000

# Learning rate progression based on your successful runs
LR_PHASES = [
    (1800, 2e-5),
    (3000, 1e-5),
    (5000, 5e-6)
]

WEIGHT_CAP = 3.0

ROOT = "/content/cuad_fresh_training"
BEST_MICRO = ROOT + "/best_micro"
BEST_MACRO = ROOT + "/best_macro"
HISTORY_PATH = ROOT + "/history.csv"

os.makedirs(ROOT, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
assert DEVICE.type == "cuda", "Enable GPU in Colab."

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

start_time = time.monotonic()

# ============================================================
# 2. REBUILD DATASETS AT 512 TOKENS
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = ContractDataset(
    train_chunks["chunk_text"].fillna("").tolist(),
    y_train,
    tokenizer,
    max_length=MAX_LENGTH
)

val_dataset = ContractDataset(
    val_chunks["chunk_text"].fillna("").tolist(),
    y_val,
    tokenizer,
    max_length=MAX_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    pin_memory=True
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Max token length:", MAX_LENGTH)

# ============================================================
# 3. FRESH MODEL
# ============================================================

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(categories),
    problem_type="multi_label_classification"
).to(DEVICE)

# ============================================================
# 4. CLASS-WEIGHTED LOSS
# ============================================================

y_train_np = np.asarray(y_train, dtype=np.float32)

positives = y_train_np.sum(axis=0)
negatives = len(y_train_np) - positives

class_weights = np.clip(
    np.sqrt(negatives / np.maximum(positives, 1)),
    1.0,
    WEIGHT_CAP
)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        class_weights,
        dtype=torch.float32,
        device=DEVICE
    )
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler("cuda")

# ============================================================
# 5. EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_model(model):
    model.eval()

    all_probs = []
    all_true = []

    for batch in val_loader:
        inputs = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in batch.items()
            if k != "labels"
        }

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**inputs).logits

        all_probs.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )
        all_true.append(batch["labels"].numpy())

    probs = np.concatenate(all_probs)
    true = np.concatenate(all_true).astype(int)

    # One global threshold: avoids rare-label overprediction
    best = None

    for threshold in np.arange(0.10, 0.801, 0.025):
        pred = (probs >= threshold).astype(int)

        micro = f1_score(
            true, pred,
            average="micro",
            zero_division=0
        )

        macro = f1_score(
            true, pred,
            average="macro",
            zero_division=0
        )

        if best is None or micro > best["micro_f1"]:
            per_class = f1_score(
                true, pred,
                average=None,
                zero_division=0
            )

            best = {
                "threshold": float(threshold),
                "micro_f1": float(micro),
                "macro_f1": float(macro),
                "precision": float(precision_score(
                    true, pred,
                    average="micro",
                    zero_division=0
                )),
                "recall": float(recall_score(
                    true, pred,
                    average="micro",
                    zero_division=0
                )),
                "categories_detected": int(
                    (per_class > 0).sum()
                ),
                "predicted_positives": int(pred.sum())
            }

    # Separately find the threshold that maximizes macro-F1,
    # subject to minimum precision and micro-F1 requirements.
    macro_candidate = None

    for threshold in np.arange(0.10, 0.801, 0.025):
        pred = (probs >= threshold).astype(int)

        micro = f1_score(
            true, pred,
            average="micro",
            zero_division=0
        )
        macro = f1_score(
            true, pred,
            average="macro",
            zero_division=0
        )
        precision = precision_score(
            true, pred,
            average="micro",
            zero_division=0
        )

        if precision < 0.35 or micro < 0.40:
            continue

        if (
            macro_candidate is None
            or macro > macro_candidate["macro_f1"]
        ):
            per_class = f1_score(
                true, pred,
                average=None,
                zero_division=0
            )

            macro_candidate = {
                "threshold": float(threshold),
                "micro_f1": float(micro),
                "macro_f1": float(macro),
                "precision": float(precision),
                "recall": float(recall_score(
                    true, pred,
                    average="micro",
                    zero_division=0
                )),
                "categories_detected": int(
                    (per_class > 0).sum()
                ),
                "predicted_positives": int(pred.sum())
            }

    model.train()
    return best, macro_candidate

# ============================================================
# 6. CHECKPOINT SAVING
# ============================================================

def save_checkpoint(path, metrics, step, objective):
    os.makedirs(path, exist_ok=True)

    model.save_pretrained(path)
    tokenizer.save_pretrained(path)

    with open(path + "/metadata.json", "w") as f:
        json.dump({
            "step": step,
            "objective": objective,
            "metrics": metrics,
            "categories": list(categories),
            "max_length": MAX_LENGTH,
            "weight_cap": WEIGHT_CAP,
            "seed": SEED
        }, f, indent=2)

# ============================================================
# 7. TRAINING
# ============================================================

best_micro_f1 = -1
best_macro_f1 = -1

history = []
step = 0
losses = []

model.train()

def learning_rate_at(step):
    for boundary, lr in LR_PHASES:
        if step < boundary:
            return lr
    return LR_PHASES[-1][1]

def run_validation(step):
    global best_micro_f1, best_macro_f1

    micro_metrics, macro_metrics = evaluate_model(model)

    row = {
        "step": step,
        "lr": learning_rate_at(step - 1),
        **micro_metrics,
        "macro_optimized_f1": (
            macro_metrics["macro_f1"]
            if macro_metrics is not None else np.nan
        )
    }
    history.append(row)

    print(
        f"\nVALIDATION @ {step} | "
        f"Micro={micro_metrics['micro_f1']:.4f} | "
        f"Macro={micro_metrics['macro_f1']:.4f} | "
        f"Precision={micro_metrics['precision']:.4f} | "
        f"Recall={micro_metrics['recall']:.4f} | "
        f"Categories={micro_metrics['categories_detected']}"
    )

    if micro_metrics["micro_f1"] > best_micro_f1:
        best_micro_f1 = micro_metrics["micro_f1"]
        save_checkpoint(
            BEST_MICRO, micro_metrics, step, "micro"
        )
        print("NEW BEST MICRO MODEL SAVED")

    if (
        macro_metrics is not None
        and macro_metrics["macro_f1"] > best_macro_f1
    ):
        best_macro_f1 = macro_metrics["macro_f1"]
        save_checkpoint(
            BEST_MACRO, macro_metrics, step, "macro"
        )
        print("NEW BEST MACRO MODEL SAVED")

    pd.DataFrame(history).to_csv(
        HISTORY_PATH, index=False
    )

while step < MAX_STEPS:

    for batch in train_loader:

        if step >= MAX_STEPS:
            break

        if time.monotonic() - start_time >= TRAIN_TIME_LIMIT:
            break

        lr = learning_rate_at(step)

        for group in optimizer.param_groups:
            group["lr"] = lr

        inputs = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in batch.items()
            if k != "labels"
        }

        labels = batch["labels"].to(
            DEVICE, non_blocking=True
        ).float()

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**inputs).logits
            loss = criterion(logits.float(), labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), 1.0
        )

        scaler.step(optimizer)
        scaler.update()

        step += 1
        losses.append(loss.item())

        if step % 100 == 0:
            print(
                f"Step {step}/{MAX_STEPS} | "
                f"LR {lr:.1e} | "
                f"Loss {np.mean(losses[-100:]):.4f} | "
                f"Elapsed "
                f"{(time.monotonic()-start_time)/60:.2f} min"
            )

        if step % EVAL_EVERY == 0:
            run_validation(step)

    if time.monotonic() - start_time >= TRAIN_TIME_LIMIT:
        break

# Final evaluation for unfinished interval
if step > 0 and step % EVAL_EVERY != 0:
    run_validation(step)

# ============================================================
# 8. FINAL RESULTS
# ============================================================

results_df = pd.DataFrame(history)

print("\n" + "=" * 60)
print("FINAL TRAINING RESULTS")
print("=" * 60)

display(results_df)

print("Best micro-F1:", round(best_micro_f1, 4))
print("Best macro-F1:", round(best_macro_f1, 4))
print("Steps completed:", step)
print(
    "Runtime:",
    round((time.monotonic()-start_time)/60, 2),
    "minutes"
)

print("\nBest overall model:", BEST_MICRO)
print("Best category-balanced model:", BEST_MACRO)
print("Training history:", HISTORY_PATH)


Training examples: 11500
Validation examples: 1998
Max token length: 512


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilbert/distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step 100/5000 | LR 2.0e-05 | Loss 0.3624 | Elapsed 0.55 min
Step 200/5000 | LR 2.0e-05 | Loss 0.2039 | Elapsed 0.77 min
Step 300/5000 | LR 2.0e-05 | Loss 0.1810 | Elapsed 0.99 min
Step 400/5000 | LR 2.0e-05 | Loss 0.1795 | Elapsed 1.21 min

VALIDATION @ 400 | Micro=0.2127 | Macro=0.0482 | Precision=0.3383 | Recall=0.1552 | Categories=5
NEW BEST MICRO MODEL SAVED
Step 500/5000 | LR 2.0e-05 | Loss 0.1817 | Elapsed 2.57 min
Step 600/5000 | LR 2.0e-05 | Loss 0.1618 | Elapsed 2.79 min
Step 700/5000 | LR 2.0e-05 | Loss 0.1705 | Elapsed 3.01 min
Step 800/5000 | LR 2.0e-05 | Loss 0.1721 | Elapsed 3.23 min

VALIDATION @ 800 | Micro=0.2439 | Macro=0.0663 | Precision=0.6519 | Recall=0.1500 | Categories=4
NEW BEST MICRO MODEL SAVED
Step 900/5000 | LR 2.0e-05 | Loss 0.1555 | Elapsed 4.59 min
Step 1000/5000 | LR 2.0e-05 | Loss 0.1627 | Elapsed 4.81 min
Step 1100/5000 | LR 2.0e-05 | Loss 0.1422 | Elapsed 5.03 min
Step 1200/5000 | LR 2.0e-05 | Loss 0.1368 | Elapsed 5.24 min

VALIDATION @ 1200 | Micro=

,step,lr,threshold,micro_f1,macro_f1,precision,recall,categories_detected,predicted_positives,macro_optimized_f1
0,400,0.00002,0.250,0.212741,0.048153,0.338290,0.155158,5,538,NaN
1,800,0.00002,0.525,0.243936,0.066284,0.651852,0.150043,4,270,NaN
2,1200,0.00002,0.300,0.395497,0.135786,0.464368,0.344416,12,870,NaN
3,1600,0.00002,0.400,0.484002,0.207212,0.513384,0.457801,16,1046,0.214954


Best micro-F1: 0.484
Best macro-F1: 0.215
Steps completed: 1600
Runtime: 9.2 minutes

Best overall model: /content/cuad_fresh_training/best_micro
Best category-balanced model: /content/cuad_fresh_training/best_macro
Training history: /content/cuad_fresh_training/history.csv


In [13]:

import os, json
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score

MODEL_PATH = "/content/cuad_fresh_training/best_micro"
OUTPUT_PATH = "/content/cuad_fresh_training/calibration"
os.makedirs(OUTPUT_PATH, exist_ok=True)

with open(MODEL_PATH + "/metadata.json") as f:
    metadata = json.load(f)

base_threshold = float(metadata["metrics"]["threshold"])

m = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
).to(DEVICE)
m.eval()

loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False
)

probs_list = []
true_list = []

with torch.no_grad():
    for batch in loader:
        inputs = {
            k: v.to(DEVICE)
            for k, v in batch.items()
            if k != "labels"
        }

        with torch.autocast("cuda", dtype=torch.float16):
            logits = m(**inputs).logits

        probs_list.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )
        true_list.append(batch["labels"].numpy())

probs = np.concatenate(probs_list)
true = np.concatenate(true_list).astype(int)

del m
torch.cuda.empty_cache()

def score(thresholds):
    pred = (probs >= thresholds).astype(int)

    return {
        "micro_f1": f1_score(
            true, pred, average="micro", zero_division=0
        ),
        "macro_f1": f1_score(
            true, pred, average="macro", zero_division=0
        ),
        "precision": precision_score(
            true, pred, average="micro", zero_division=0
        ),
        "recall": recall_score(
            true, pred, average="micro", zero_division=0
        ),
        "categories_detected": int(
            (f1_score(
                true, pred, average=None, zero_division=0
            ) > 0).sum()
        ),
        "predicted_positives": int(pred.sum())
    }

# Coordinate-descent optimization:
# optimize the combined objective rather than each class alone.
#
# Constraints:
# - micro-F1 cannot fall more than 0.02 below baseline
# - total predictions cannot exceed 1.5x baseline
# - minimum 10 validation positives before class-specific tuning

base = np.full(len(categories), base_threshold)
base_metrics = score(base)

thresholds = base.copy()

def objective(metrics):
    if metrics["micro_f1"] < base_metrics["micro_f1"] - 0.02:
        return -1e9

    if metrics["predicted_positives"] > (
        base_metrics["predicted_positives"] * 1.5
    ):
        return -1e9

    return (
        0.65 * metrics["micro_f1"]
        + 0.35 * metrics["macro_f1"]
    )

candidates = np.arange(0.15, 0.751, 0.025)

for iteration in range(3):
    for j in range(len(categories)):

        if true[:, j].sum() < 10:
            continue

        current_best = objective(score(thresholds))
        best_t = thresholds[j]

        for t in candidates:
            trial = thresholds.copy()
            trial[j] = t

            value = objective(score(trial))

            if value > current_best + 1e-8:
                current_best = value
                best_t = t

        thresholds[j] = best_t

    print(
        f"Iteration {iteration+1}:",
        score(thresholds)
    )

final_metrics = score(thresholds)

print("\nBASELINE:", base_metrics)
print("CALIBRATED:", final_metrics)

threshold_table = pd.DataFrame({
    "category": categories,
    "threshold": thresholds,
    "validation_positives": true.sum(axis=0)
})

display(threshold_table)

with open(OUTPUT_PATH + "/thresholds.json", "w") as f:
    json.dump({
        "model_path": MODEL_PATH,
        "thresholds": thresholds.tolist(),
        "metrics": final_metrics
    }, f, indent=2)

threshold_table.to_csv(
    OUTPUT_PATH + "/thresholds.csv",
    index=False
)

print("\nSaved calibration to:", OUTPUT_PATH)


Iteration 1: {'micro_f1': 0.5535296867695184, 'macro_f1': 0.30911808403401825, 'precision': 0.6128364389233955, 'recall': 0.5046888320545609, 'categories_detected': 23, 'predicted_positives': 966}
Iteration 2: {'micro_f1': 0.5534709193245778, 'macro_f1': 0.3093779184237835, 'precision': 0.6152241918665277, 'recall': 0.5029838022165388, 'categories_detected': 23, 'predicted_positives': 959}
Iteration 3: {'micro_f1': 0.5534709193245778, 'macro_f1': 0.3093779184237835, 'precision': 0.6152241918665277, 'recall': 0.5029838022165388, 'categories_detected': 23, 'predicted_positives': 959}

BASELINE: {'micro_f1': 0.48400180261379, 'macro_f1': 0.2072117482841476, 'precision': 0.5133843212237094, 'recall': 0.4578005115089514, 'categories_detected': 16, 'predicted_positives': 1046}
CALIBRATED: {'micro_f1': 0.5534709193245778, 'macro_f1': 0.3093779184237835, 'precision': 0.6152241918665277, 'recall': 0.5029838022165388, 'categories_detected': 23, 'predicted_positives': 959}


,category,threshold,validation_positives
0,Affiliate License-Licensee,0.400,18
1,Affiliate License-Licensor,0.400,3
2,Agreement Date,0.725,55
3,Anti-Assignment,0.675,67
4,Audit Rights,0.400,71
5,Cap On Liability,0.725,57
6,Change Of Control,0.400,29
7,Competitive Restriction Exception,0.400,10
8,Covenant Not To Sue,0.400,17
9,Document Name,0.750,57



Saved calibration to: /content/cuad_fresh_training/calibration


In [14]:

import os
import json
import shutil
from google.colab import files

ROOT = "/content/cuad_fresh_training"
MODEL_PATH = f"{ROOT}/best_micro"
CALIBRATION_PATH = f"{ROOT}/calibration/thresholds.json"

EXPORT_DIR = "/content/CUAD_Classifier_v1"
ZIP_PATH = "/content/CUAD_Classifier_v1.zip"

# Verify required files
assert os.path.isdir(MODEL_PATH), "Model folder not found"
assert os.path.isfile(CALIBRATION_PATH), "Calibration file not found"

# Recreate clean export folder
if os.path.exists(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)

os.makedirs(EXPORT_DIR)

# Copy trained model and tokenizer
shutil.copytree(
    MODEL_PATH,
    f"{EXPORT_DIR}/model"
)

# Copy calibrated thresholds
shutil.copy2(
    CALIBRATION_PATH,
    f"{EXPORT_DIR}/thresholds.json"
)

# Copy training history
history_path = f"{ROOT}/history.csv"

if os.path.exists(history_path):
    shutil.copy2(
        history_path,
        f"{EXPORT_DIR}/training_history.csv"
    )

# Create model information
with open(CALIBRATION_PATH) as f:
    calibration = json.load(f)

with open(f"{MODEL_PATH}/metadata.json") as f:
    metadata = json.load(f)

model_info = {
    "model_name": "CUAD Multi-Label Contract Classifier",
    "version": "1.0",
    "base_model": "distilbert/distilroberta-base",
    "num_categories": len(categories),
    "max_length": 512,
    "categories": list(categories),
    "validation_metrics": calibration["metrics"],
    "calibration_method": "Per-category thresholds",
    "training_checkpoint": metadata,
    "notes": (
        "Metrics are from validation data used for "
        "threshold calibration, not independent test data."
    )
}

with open(f"{EXPORT_DIR}/model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)

# Zip everything
shutil.make_archive(
    "/content/CUAD_Classifier_v1",
    "zip",
    EXPORT_DIR
)

size_mb = os.path.getsize(ZIP_PATH) / (1024 ** 2)

print("MODEL EXPORTED SUCCESSFULLY")
print(f"ZIP size: {size_mb:.1f} MB")
print("Location:", ZIP_PATH)
print("Micro-F1:", calibration["metrics"]["micro_f1"])
print("Macro-F1:", calibration["metrics"]["macro_f1"])

# Download to your computer
files.download(ZIP_PATH)


MODEL EXPORTED SUCCESSFULLY
ZIP size: 292.3 MB
Location: /content/CUAD_Classifier_v1.zip
Micro-F1: 0.5534709193245778
Macro-F1: 0.3093779184237835


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:

import json
import numpy as np
import torch

from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import (
    f1_score, precision_score, recall_score
)

EXPORT_DIR = "/content/CUAD_Classifier_v1"

with open(f"{EXPORT_DIR}/thresholds.json") as f:
    thresholds = np.array(
        json.load(f)["thresholds"]
    )

with open(f"{EXPORT_DIR}/model_info.json") as f:
    info = json.load(f)

assert list(categories) == info["categories"], (
    "Category order does not match the saved model"
)

# Rebuild test data at 512 tokens
test_dataset_512 = ContractDataset(
    test_chunks["chunk_text"].fillna("").tolist(),
    y_test,
    tokenizer,
    max_length=512
)

test_loader = DataLoader(
    test_dataset_512,
    batch_size=32,
    shuffle=False
)

test_model = AutoModelForSequenceClassification.from_pretrained(
    f"{EXPORT_DIR}/model"
).to(DEVICE)

test_model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        inputs = {
            k: v.to(DEVICE)
            for k, v in batch.items()
            if k != "labels"
        }

        with torch.autocast("cuda", dtype=torch.float16):
            logits = test_model(**inputs).logits

        all_probs.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )
        all_labels.append(batch["labels"].numpy())

probs = np.concatenate(all_probs)
true = np.concatenate(all_labels).astype(int)

assert probs.shape[1] == len(thresholds)

pred = (probs >= thresholds[None, :]).astype(int)

print("FINAL TEST RESULTS")
print("=" * 45)

print("Micro-F1:", f1_score(
    true, pred, average="micro", zero_division=0
))
print("Macro-F1:", f1_score(
    true, pred, average="macro", zero_division=0
))
print("Precision:", precision_score(
    true, pred, average="micro", zero_division=0
))
print("Recall:", recall_score(
    true, pred, average="micro", zero_division=0
))

per_class = f1_score(
    true, pred, average=None, zero_division=0
)

print(
    "Categories detected:",
    int((per_class > 0).sum()),
    "/",
    len(categories)
)


FINAL TEST RESULTS
Micro-F1: 0.5640074211502782
Macro-F1: 0.3016371720710894
Precision: 0.6368715083798883
Recall: 0.5061043285238623
Categories detected: 23 / 41


In [16]:

import json
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score
)

paths = {
    "best_micro": "/content/cuad_fresh_training/best_micro",
    "best_macro": "/content/cuad_fresh_training/best_macro"
}

loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False
)

@torch.no_grad()
def get_predictions(path):
    m = AutoModelForSequenceClassification.from_pretrained(
        path
    ).to(DEVICE)
    m.eval()

    probs = []
    true = []

    for batch in loader:
        inputs = {
            k: v.to(DEVICE)
            for k, v in batch.items()
            if k != "labels"
        }

        with torch.autocast("cuda", dtype=torch.float16):
            logits = m(**inputs).logits

        probs.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )
        true.append(batch["labels"].numpy())

    del m
    torch.cuda.empty_cache()

    return np.concatenate(probs), np.concatenate(true).astype(int)

for name, path in paths.items():
    with open(path + "/metadata.json") as f:
        metadata = json.load(f)

    threshold = metadata["metrics"]["threshold"]

    probs, true = get_predictions(path)
    pred = (probs >= threshold).astype(int)

    rows = []

    for j, category in enumerate(categories):
        positives = int(true[:, j].sum())

        ap = (
            average_precision_score(true[:, j], probs[:, j])
            if positives > 0 else np.nan
        )

        rows.append({
            "category": category,
            "actual": positives,
            "predicted": int(pred[:, j].sum()),
            "precision": precision_score(
                true[:, j], pred[:, j], zero_division=0
            ),
            "recall": recall_score(
                true[:, j], pred[:, j], zero_division=0
            ),
            "f1": f1_score(
                true[:, j], pred[:, j], zero_division=0
            ),
            "average_precision": ap,
            "mean_positive_probability": (
                float(probs[true[:, j] == 1, j].mean())
                if positives > 0 else np.nan
            ),
            "mean_negative_probability": (
                float(probs[true[:, j] == 0, j].mean())
                if (true[:, j] == 0).any() else np.nan
            )
        })

    df = pd.DataFrame(rows).sort_values(
        ["f1", "average_precision"],
        ascending=[True, False]
    )

    print(f"\n{name.upper()} — threshold {threshold:.3f}")
    print("Micro-F1:", round(
        f1_score(true, pred, average="micro"), 4
    ))
    print("Macro-F1:", round(
        f1_score(true, pred, average="macro"), 4
    ))

    display(df)

    df.to_csv(
        f"/content/cuad_{name}_category_diagnostics.csv",
        index=False
    )



BEST_MICRO — threshold 0.400
Micro-F1: 0.484
Macro-F1: 0.2072


,category,actual,predicted,precision,recall,f1,average_precision,mean_positive_probability,mean_negative_probability
16,Irrevocable Or Perpetual License,23,0,0.000000,0.000000,0.000000,0.702353,0.174692,0.023356
0,Affiliate License-Licensee,18,0,0.000000,0.000000,0.000000,0.594333,0.111445,0.020175
27,Notice Period To Terminate Renewal,11,0,0.000000,0.000000,0.000000,0.542604,0.211140,0.022945
15,Ip Ownership Assignment,28,0,0.000000,0.000000,0.000000,0.393512,0.134891,0.048360
35,Termination For Convenience,27,0,0.000000,0.000000,0.000000,0.362987,0.267775,0.047770
40,Warranty Duration,19,0,0.000000,0.000000,0.000000,0.261712,0.092293,0.038469
11,Exclusivity,45,0,0.000000,0.000000,0.000000,0.226849,0.203925,0.059327
20,Minimum Commitment,39,1,0.000000,0.000000,0.000000,0.125628,0.156709,0.070500
6,Change Of Control,29,0,0.000000,0.000000,0.000000,0.104845,0.091587,0.039953
39,Volume Restriction,18,0,0.000000,0.000000,0.000000,0.048589,0.068794,0.032357



BEST_MACRO — threshold 0.350
Micro-F1: 0.4746
Macro-F1: 0.215


,category,actual,predicted,precision,recall,f1,average_precision,mean_positive_probability,mean_negative_probability
16,Irrevocable Or Perpetual License,23,0,0.000000,0.000000,0.000000,0.702353,0.174692,0.023356
0,Affiliate License-Licensee,18,0,0.000000,0.000000,0.000000,0.594333,0.111445,0.020175
27,Notice Period To Terminate Renewal,11,0,0.000000,0.000000,0.000000,0.542604,0.211140,0.022945
15,Ip Ownership Assignment,28,0,0.000000,0.000000,0.000000,0.393512,0.134891,0.048360
40,Warranty Duration,19,0,0.000000,0.000000,0.000000,0.261712,0.092293,0.038469
20,Minimum Commitment,39,1,0.000000,0.000000,0.000000,0.125628,0.156709,0.070500
6,Change Of Control,29,0,0.000000,0.000000,0.000000,0.104845,0.091587,0.039953
39,Volume Restriction,18,0,0.000000,0.000000,0.000000,0.048589,0.068794,0.032357
33,Rofr/Rofo/Rofn,19,0,0.000000,0.000000,0.000000,0.047460,0.094707,0.049593
19,Liquidated Damages,4,0,0.000000,0.000000,0.000000,0.042028,0.055938,0.024417


In [21]:

import json
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    precision_recall_fscore_support
)

BEST_PATH = "/content/cuad_deberta_fast/best_model"

with open(f"{BEST_PATH}/metadata.json") as f:
    metadata = json.load(f)

threshold = metadata["metrics"]["threshold"]

tokenizer = AutoTokenizer.from_pretrained(BEST_PATH)
model = AutoModelForSequenceClassification.from_pretrained(
    BEST_PATH
).to(DEVICE).eval()

# Re-tokenize validation data for this exact checkpoint
texts = val_chunks["chunk_text"].fillna("").tolist()
probs_list = []

with torch.no_grad():
    for i in range(0, len(texts), 16):
        batch = tokenizer(
            texts[i:i+16],
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**batch).logits

        probs_list.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )

probs = np.concatenate(probs_list)
true = np.asarray(y_val).astype(int)

def calculate_metrics(probs, true, t):
    pred = (probs >= t).astype(int)
    return {
        "micro_f1": f1_score(true, pred, average="micro"),
        "macro_f1": f1_score(true, pred, average="macro"),
        "precision": precision_score(true, pred, average="micro", zero_division=0),
        "recall": recall_score(true, pred, average="micro", zero_division=0)
    }

for t in [0.5, threshold]:
    print(f"Threshold {t:.3f}:",
          calculate_metrics(probs, true, t))

predictions = (probs >= threshold).astype(int)

p, r, f1, support = precision_recall_fscore_support(
    true, predictions, average=None, zero_division=0
)

per_category = pd.DataFrame({
    "category": categories,
    "precision": p,
    "recall": r,
    "f1": f1,
    "actual_count": support,
    "predicted_count": predictions.sum(axis=0)
})

display(per_category.sort_values("f1", ascending=False).head(15))
display(per_category.sort_values("f1").head(15))


The tokenizer you are loading from '/content/cuad_deberta_fast/best_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Threshold 0.500: {'micro_f1': 0.26469085334695963, 'macro_f1': 0.06753805004107605, 'precision': 0.33035714285714285, 'recall': 0.2208013640238704}
Threshold 0.450: {'micro_f1': 0.2922289436947417, 'macro_f1': 0.07651640398274148, 'precision': 0.32172131147540983, 'recall': 0.26768968456947995}


,category,precision,recall,f1,actual_count,predicted_count
18,License Grant,0.612245,0.750000,0.674157,80,98
28,Parties,0.402685,0.705882,0.512821,85,149
9,Document Name,0.311258,0.824561,0.451923,57,151
2,Agreement Date,0.314286,0.800000,0.451282,55,140
3,Anti-Assignment,0.380282,0.402985,0.391304,67,71
10,Effective Date,0.229508,0.595745,0.331361,47,122
13,Governing Law,0.195918,0.941176,0.324324,51,245
1,Affiliate License-Licensor,0.000000,0.000000,0.000000,3,0
0,Affiliate License-Licensee,0.000000,0.000000,0.000000,18,0
8,Covenant Not To Sue,0.000000,0.000000,0.000000,17,0


,category,precision,recall,f1,actual_count,predicted_count
0,Affiliate License-Licensee,0.0,0.0,0.0,18,0
1,Affiliate License-Licensor,0.0,0.0,0.0,3,0
5,Cap On Liability,0.0,0.0,0.0,57,0
4,Audit Rights,0.0,0.0,0.0,71,0
6,Change Of Control,0.0,0.0,0.0,29,0
7,Competitive Restriction Exception,0.0,0.0,0.0,10,0
11,Exclusivity,0.0,0.0,0.0,45,0
8,Covenant Not To Sue,0.0,0.0,0.0,17,0
15,Ip Ownership Assignment,0.0,0.0,0.0,28,0
12,Expiration Date,0.0,0.0,0.0,57,0


In [22]:

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

# Assumes 'probs' and 'true' are from the full validation set
assert probs.shape == true.shape
assert probs.shape[1] == len(categories)

global_threshold = threshold
thresholds = np.full(len(categories), global_threshold)

# Only tune categories with enough validation positives.
# Otherwise retain the global threshold to reduce overfitting.
for j in range(len(categories)):
    if true[:, j].sum() < 10:
        continue

    best_f1 = -1
    best_t = global_threshold

    for t in np.arange(0.05, 0.81, 0.025):
        pred = (probs[:, j] >= t).astype(int)

        score = f1_score(
            true[:, j],
            pred,
            zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_t = float(t)

    thresholds[j] = best_t

pred_global = (probs >= global_threshold).astype(int)
pred_tuned = (probs >= thresholds[None, :]).astype(int)

for name, pred in [
    ("Global threshold", pred_global),
    ("Per-category thresholds", pred_tuned)
]:
    print("\n", name)
    print("Micro-F1:", round(
        f1_score(true, pred, average="micro", zero_division=0), 4
    ))
    print("Macro-F1:", round(
        f1_score(true, pred, average="macro", zero_division=0), 4
    ))
    print("Precision:", round(
        precision_score(true, pred, average="micro", zero_division=0), 4
    ))
    print("Recall:", round(
        recall_score(true, pred, average="micro", zero_division=0), 4
    ))
    print("Predicted positives:", int(pred.sum()))

threshold_df = pd.DataFrame({
    "category": categories,
    "threshold": thresholds,
    "validation_positives": true.sum(axis=0)
})

display(threshold_df)

threshold_df.to_csv(
    "/content/cuad_category_thresholds.csv",
    index=False
)



 Global threshold
Micro-F1: 0.2922
Macro-F1: 0.0765
Precision: 0.3217
Recall: 0.2677
Predicted positives: 976

 Per-category thresholds
Micro-F1: 0.1389
Macro-F1: 0.165
Precision: 0.0792
Recall: 0.5618
Predicted positives: 8316


,category,threshold,validation_positives
0,Affiliate License-Licensee,0.100,18
1,Affiliate License-Licensor,0.450,3
2,Agreement Date,0.425,55
3,Anti-Assignment,0.450,67
4,Audit Rights,0.100,71
5,Cap On Liability,0.300,57
6,Change Of Control,0.125,29
7,Competitive Restriction Exception,0.050,10
8,Covenant Not To Sue,0.100,17
9,Document Name,0.625,57


Test predictions

In [23]:

# EDIT THESE VALUES TO MATCH ADVISOR DECISIONS

RISK_CONFIG = {
    "base_score": 0,
    "max_score": 100,
    "review_threshold": 0.50,

    # Risk points added when a clause is detected
    "weights": {
        "Uncapped Liability": 35,
        "Non-Compete": 25,
        "Exclusivity": 15,
        "Anti-Assignment": 10,
        "Termination For Convenience": 15,
        "Ip Ownership Assignment": 20,
        "Minimum Commitment": 15,
        "Liquidated Damages": 20
    },

    # High-risk overrides: minimum final score
    "overrides": {
        "Uncapped Liability": 85,
        "Non-Compete": 75
    },

    # Missing important clauses add risk
    "required_clauses": {
        "Governing Law": 10,
        "Cap On Liability": 25,
        "Termination For Convenience": 10,
        "Audit Rights": 5
    },

    "missing_requires_review": True
}

In [24]:

def calculate_risk(detected, config=RISK_CONFIG):
    """
    detected: dictionary mapping clause names
              to classifier probabilities.
    """
    score = config["base_score"]
    explanations = []
    overrides = []
    review = []

    threshold = config["review_threshold"]

    # Weighted risk signals
    for clause, weight in config["weights"].items():
        confidence = detected.get(clause, 0)

        if confidence >= threshold:
            score += weight
            explanations.append({
                "signal": clause,
                "type": "detected_risk",
                "confidence": round(float(confidence), 3),
                "points": weight
            })

    # Missing important clauses
    for clause, penalty in config["required_clauses"].items():
        confidence = detected.get(clause, 0)

        if confidence < threshold:
            score += penalty
            explanations.append({
                "signal": clause,
                "type": "not_detected",
                "confidence": round(float(confidence), 3),
                "points": penalty
            })

            if config["missing_requires_review"]:
                review.append(clause)

    # High-risk overrides
    for clause, minimum_score in config["overrides"].items():
        confidence = detected.get(clause, 0)

        if confidence >= threshold:
            previous = score
            score = max(score, minimum_score)

            overrides.append({
                "clause": clause,
                "minimum_score": minimum_score,
                "applied": score > previous
            })

    score = min(config["max_score"], score)

    level = (
        "HIGH" if score >= 70
        else "MEDIUM" if score >= 35
        else "LOW"
    )

    return {
        "risk_score": score,
        "risk_level": level,
        "explanations": explanations,
        "overrides": overrides,
        "missing_clause_review": review
    }


In [25]:

sample_predictions = {
    "Uncapped Liability": 0.92,
    "Non-Compete": 0.81,
    "Exclusivity": 0.73,
    "Governing Law": 0.94,
    "Cap On Liability": 0.12,
    "Termination For Convenience": 0.85,
    "Audit Rights": 0.91
}

result = calculate_risk(sample_predictions)

import json
print(json.dumps(result, indent=2))


{
  "risk_score": 100,
  "risk_level": "HIGH",
  "explanations": [
    {
      "signal": "Uncapped Liability",
      "type": "detected_risk",
      "confidence": 0.92,
      "points": 35
    },
    {
      "signal": "Non-Compete",
      "type": "detected_risk",
      "confidence": 0.81,
      "points": 25
    },
    {
      "signal": "Exclusivity",
      "type": "detected_risk",
      "confidence": 0.73,
      "points": 15
    },
    {
      "signal": "Termination For Convenience",
      "type": "detected_risk",
      "confidence": 0.85,
      "points": 15
    },
    {
      "signal": "Cap On Liability",
      "type": "not_detected",
      "confidence": 0.12,
      "points": 25
    }
  ],
  "overrides": [
    {
      "clause": "Uncapped Liability",
      "minimum_score": 85,
      "applied": false
    },
    {
      "clause": "Non-Compete",
      "minimum_score": 75,
      "applied": false
    }
  ],
  "missing_clause_review": [
    "Cap On Liability"
  ]
}


Save the model

In [26]:

SAVE_PATH = "./cuad_distilroberta"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully!")


Model saved successfully!


In [27]:

import json
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_DIR = "/content/CUAD_Classifier_v1"

with open(f"{MODEL_DIR}/model_info.json") as f:
    model_info = json.load(f)

with open(f"{MODEL_DIR}/thresholds.json") as f:
    saved_thresholds = json.load(f)["thresholds"]

risk_categories = model_info["categories"]

risk_tokenizer = AutoTokenizer.from_pretrained(
    f"{MODEL_DIR}/model"
)

risk_model = AutoModelForSequenceClassification.from_pretrained(
    f"{MODEL_DIR}/model"
).to(DEVICE).eval()

def score_contract(text):
    tokens = risk_tokenizer(
        text, add_special_tokens=False
    )["input_ids"]

    # 510 content tokens + 2 special tokens
    chunks = [
        risk_tokenizer.decode(
            tokens[i:i+510],
            skip_special_tokens=True
        )
        for i in range(0, max(len(tokens), 1), 400)
    ]

    probabilities = []

    for chunk in chunks:
        inputs = risk_tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(DEVICE)

        with torch.no_grad():
            logits = risk_model(**inputs).logits

        probabilities.append(
            torch.sigmoid(logits)[0].cpu().numpy()
        )

    # Aggregate evidence across chunks
    max_probs = np.max(probabilities, axis=0)

    # Respect the model's calibrated thresholds
    normalized = {
        category: float(max_probs[j])
        for j, category in enumerate(risk_categories)
        if max_probs[j] >= saved_thresholds[j]
    }

    # The scoring engine uses its own threshold;
    # mark calibrated detections as present.
    detected = {name: 1.0 for name in normalized}

    result = calculate_risk(detected)
    result["detected_clauses"] = normalized
    result["chunks_analyzed"] = len(chunks)

    return result

sample_contract = """
This agreement contains uncapped liability.
The contractor agrees not to compete for two years.
This agreement is governed by Texas law.
"""

print(json.dumps(score_contract(sample_contract), indent=2))


{
  "risk_score": 40,
  "risk_level": "MEDIUM",
  "explanations": [
    {
      "signal": "Cap On Liability",
      "type": "not_detected",
      "confidence": 0.0,
      "points": 25
    },
    {
      "signal": "Termination For Convenience",
      "type": "not_detected",
      "confidence": 0.0,
      "points": 10
    },
    {
      "signal": "Audit Rights",
      "type": "not_detected",
      "confidence": 0.0,
      "points": 5
    }
  ],
  "overrides": [],
  "missing_clause_review": [
    "Cap On Liability",
    "Termination For Convenience",
    "Audit Rights"
  ],
  "detected_clauses": {
    "Governing Law": 0.7795121669769287
  },
  "chunks_analyzed": 1
}
